In [1]:
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-hdq7fqxb
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-hdq7fqxb
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'segment_anything' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'segment_anything'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36636 sha256=e9b47995b8276b5e5053c64

In [2]:
import os, cv2, numpy as np
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

CKPT = "checkpoints/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
IMG_DIR = "seg_test_images"
OUT_DIR = "outputs_sam"
os.makedirs(OUT_DIR, exist_ok=True)

# 1) Load model + automatic mask generator (good defaults for food)
sam = sam_model_registry[MODEL_TYPE](checkpoint=CKPT)
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,          # denser proposals (increase for complex plates)
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    min_mask_region_area=2000,   # filter tiny crumbs; lower if you want every bit
)

def overlay_masks(img_bgr, masks):
    overlay = img_bgr.copy()
    for i, m in enumerate(masks):
        mask = m["segmentation"].astype(np.uint8)
        color = np.random.randint(0,255,3, dtype=np.uint8).tolist()
        # color overlay
        overlay[mask==1] = (0.5*overlay[mask==1] + 0.5*np.array(color)).astype(np.uint8)
        # boundary
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, (255,255,255), 2)
    return overlay

for name in os.listdir(IMG_DIR):
    if not name.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp")):
        continue
    path = os.path.join(IMG_DIR, name)
    img = cv2.imread(path)
    masks = mask_generator.generate(img)

    # Sort largest first (useful to export top food items)
    masks = sorted(masks, key=lambda m: m["area"], reverse=True)

    # Save overlay
    overlay = overlay_masks(img, masks)
    cv2.imwrite(os.path.join(OUT_DIR, f"{os.path.splitext(name)[0]}_overlay.jpg"), overlay)

    # Save per-instance crop PNGs
    base = os.path.splitext(name)[0]
    item_dir = os.path.join(OUT_DIR, f"{base}_items")
    os.makedirs(item_dir, exist_ok=True)
    for i, m in enumerate(masks[:10]):  # export top-10 regions (tweak as you like)
        mask = m["segmentation"].astype(np.uint8)
        x,y,w,h = cv2.boundingRect(mask)
        crop = img[y:y+h, x:x+w].copy()
        crop[mask[y:y+h, x:x+w]==0] = 0  # keep only the food region
        cv2.imwrite(os.path.join(item_dir, f"{base}_item_{i+1}.png"), crop)

print("Done. Check the outputs_sam/ folder.")


/home/chahar/miniconda3/envs/food_cal/lib/python3.11/site-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f

Done. Check the outputs_sam/ folder.


In [3]:
import sys, torch, torchvision
print("Python:", sys.version)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available(), "CUDA:", torch.version.cuda)

Python: 3.11.9 (main, Apr 19 2024, 16:48:06) [GCC 11.2.0]
torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
CUDA available: True CUDA: 12.1
